In this notebook we show how to pull the available training feature matrix that was used to train the stem.MAP model using the datasets module from HuggingFace. This is just an example of how to use the workflow. Alternatively, the user could incorporate additional features into the feature matrix and train their own model. The user can also use newer versions of AutoGluon for their own training purposes. See the latest [AutoGluon documentation](https://auto.gluon.ai/stable/api/autogluon.tabular.TabularPredictor.html) for additional information on parameters that can be useful for model building

In [1]:
# packages #

#sfisch6: remember that version 0.8.2 is needed to use our stem.MAP model, but if training a new model from scratch
#sfisch6: the user can utilize newer releases of AutoGluon 
from autogluon.tabular import TabularPredictor

#sfisch6: huggingface module which we will use to pull the training feature matrix
from datasets import load_dataset 

In [7]:
#sfisch6: temporary line while the repo is still private
from getpass import getpass
from huggingface_hub import login

login(token=getpass("Hugging Face token: "))

Hugging Face token: ········


In [9]:
#sfisch6: pull train fmat from huggingface
dataset = load_dataset('sfisch/stem.MAP', token=True)
df_train = dataset["train"].to_pandas()

#sfisch6: first col contains the UniProt accessions for each pair
df_train.head()

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

,ID,pearsonR_MES_IEX_mixedbed_20180212,pearsonR_Mallam_MES_SEC_RNAse,pearsonR_Mallam_MES_control,humap3_pearsonR_MES_IEX_mixedbed_20180212,humap3_pearsonR_Mallam_MES_SEC_RNAse,humap3_pearsonR_Mallam_MES_control,label
0,P54728 P22933,0.0000,0.0560,-0.0300,0.000000,0.000429,-0.000230,-1
1,Q99J62 Q9CWF6,0.0000,-0.0608,0.2059,0.000000,-0.001270,0.004302,-1
2,Q8R2R9 Q9EQ28,0.0000,-0.0134,0.5210,0.000000,-0.000184,0.007172,-1
3,P62315 P54726,0.1133,0.3514,0.0146,0.000301,0.000935,0.000039,-1
4,Q01149 Q9R190,-0.0135,-0.0384,0.1120,0.000004,0.000005,0.000008,-1


In [10]:
df_train.shape

(20558, 8)

In [11]:
df_train.columns

Index(['ID', 'pearsonR_MES_IEX_mixedbed_20180212',
       'pearsonR_Mallam_MES_SEC_RNAse', 'pearsonR_Mallam_MES_control',
       'humap3_pearsonR_MES_IEX_mixedbed_20180212',
       'humap3_pearsonR_Mallam_MES_SEC_RNAse',
       'humap3_pearsonR_Mallam_MES_control', 'label'],
      dtype='object')

In [18]:
df_train['label'].value_counts()

-1    17654
 1     2904
Name: label, dtype: int64

In [13]:
#sfisch6: we drop the ID column because we do not want accessions used for training
train_trim = df_train.drop(["ID"], axis=1)

In [14]:
#sfisch6: define label column to pass to model for training - note the model will expect the same label column
# in training and testing
label = 'label'

In [15]:
#sfisch6: define folder to store trained models; one folder per model is needed
# save_path = './your_save_path_here'
save_path = './stem.MAP_newtraining_20261013/' 

In [16]:
#sfisch6: train model by passing training feature matrix, label col, and path for saved models
# different evaluation metrics 
predictor = TabularPredictor(label=label, eval_metric="balanced_accuracy", path=save_path).fit(train_trim)

/home/jupyter-sfisch6_2025/.local/lib/python3.9/site-packages/autogluon/common/utils/utils.py:74: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Beginning AutoGluon training ...
AutoGluon will save models to "./stem.MAP_newtraining_20261013/"
AutoGluon Version:  0.8.2
Python Version:     3.9.7
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #167~18.04.1-Ubuntu SMP Wed May 24 00:51:42 UTC 2023
Disk Space Avail:   119.76 GB / 2009.04 GB (6.0%)
Train Data Rows:    20558
Train Data Columns: 6
Label Column: label
Preprocessing data ...
AutoGluon infers your prediction problem is: 'binary' (because only two unique label-values observed).
	2 unique label values:  [-1, 1]
	If 'binary' is not the correct problem_type, please manually specify the problem_ty

In [17]:
#sfisch6: Example of model inspection - leaderboard shows you the models trained and their performance
predictor.leaderboard()

                 model  score_val  pred_time_val  fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        LightGBMLarge   1.000000       0.005824  1.400830                0.005824           1.400830            1       True          9
1     RandomForestEntr   1.000000       0.158998  1.025141                0.158998           1.025141            1       True          6
2  WeightedEnsemble_L2   1.000000       0.161566  1.628684                0.002568           0.603543            2       True         10
3             LightGBM   0.996552       0.005865  0.783675                0.005865           0.783675            1       True          4
4     RandomForestGini   0.996552       0.175908  1.106328                0.175908           1.106328            1       True          5
5       ExtraTreesEntr   0.996552       0.189660  1.437347                0.189660           1.437347            1       True          8
6       ExtraTreesGini   0.996552       0

,model,score_val,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMLarge,1.000000,0.005824,1.400830,0.005824,1.400830,1,True,9
1,RandomForestEntr,1.000000,0.158998,1.025141,0.158998,1.025141,1,True,6
2,WeightedEnsemble_L2,1.000000,0.161566,1.628684,0.002568,0.603543,2,True,10
3,LightGBM,0.996552,0.005865,0.783675,0.005865,0.783675,1,True,4
4,RandomForestGini,0.996552,0.175908,1.106328,0.175908,1.106328,1,True,5
5,ExtraTreesEntr,0.996552,0.189660,1.437347,0.189660,1.437347,1,True,8
6,ExtraTreesGini,0.996552,0.190891,1.458225,0.190891,1.458225,1,True,7
7,KNeighborsDist,0.989655,0.064517,0.026260,0.064517,0.026260,1,True,2
8,LightGBMXT,0.970715,0.005879,0.922119,0.005879,0.922119,1,True,3
9,KNeighborsUnif,0.950025,0.054481,0.021972,0.054481,0.021972,1,True,1
